## Image-to-Image Translation with Conditional Adversarial Networks (Pix2Pix)

This notebook provides an implementation of [Pix2Pix](https://arxiv.org/abs/1611.07004) image-to-image translation. 

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os
import glob
from PIL import Image

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.optim import Adam
import torch.autograd as autograd
from torch.autograd import Variable
from torch.utils.data import Dataset
# Torchvision
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torch.nn.functional as F
# tqdm
from tqdm.notebook import tqdm, trange
from torchsummary import summary

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Download and Prepare Facades Dataset

%%bash
URL=http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz
TAR_FILE=./data/facades.tar.gz
TARGET_DIR=./data/facades/
wget -N $URL -O $TAR_FILE
mkdir -p $TARGET_DIR
tar -zxvf $TAR_FILE -C ./data/
rm $TAR_FILE

In [ ]:
train_dir = 'data/facades/train/'
test_dir = 'data/facades/test/'

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, root, transforms_=None,):
        self.transform = transforms_
        self.files = sorted(glob.glob("%s/*.jpg" % root))

    def __getitem__(self, index):
        img = Image.open(self.files[index % len(self.files)])
        img = self.transform(img)
        return img

    def __len__(self):
        return len(self.files)

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

In [ ]:
train_batch_size = 16
train_loader = data.DataLoader(CustomImageDataset(train_dir, transforms_=transform), 
                               batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True,
                               num_workers=4)

In [ ]:
test_batch_size = 16
test_loader = data.DataLoader(CustomImageDataset(test_dir, transforms_=transform),
                               batch_size=test_batch_size, 
                               shuffle=False,
                               num_workers=1)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=4, 
                       nrow=2,
                       filename=""):
    image_grid = make_grid(image_tensor[:num_images].detach().cpu(), nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Generator class
The generator has a U-Net structure, that is encoder-decoder form with skip connections between the corresponding levels in encoder and decoder.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.down1 = self.downsample_block(3, 64, normalize=False)
        self.down2 = self.downsample_block(64, 128)
        self.down3 = self.downsample_block(128, 256)
        self.down4 = self.downsample_block(256, 512, dropout=0.5)
        self.down5 = self.downsample_block(512, 512, dropout=0.5)
        self.down6 = self.downsample_block(512, 512, dropout=0.5)
        self.down7 = self.downsample_block(512, 512, dropout=0.5)
        self.down8 = self.downsample_block(512, 512, normalize=False, dropout=0.5)
        
        self.up8 = self.upsample_block(512, 512, dropout=0.5)
        self.up7 = self.upsample_block(1024, 512, dropout=0.5)
        self.up6 = self.upsample_block(1024, 512, dropout=0.5)
        self.up5 = self.upsample_block(1024, 512, dropout=0.5)
        self.up4 = self.upsample_block(1024, 256)
        self.up3 = self.upsample_block(512, 128)
        self.up2 = self.upsample_block(256, 64)
        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(128, 3, 4, padding=1),
            nn.Sigmoid(),
        )
        
    def downsample_block(self, in_channels, out_channels,
                        normalize=True, dropout=0.0):
        layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2))
        if dropout:
            layers.append(nn.Dropout(dropout))
        return nn.Sequential(*layers)
    
    def upsample_block(self, in_channels, out_channels, dropout=0.0):
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        return nn.Sequential(*layers)
    
    def forward(self, image):
        dn1 = self.down1(image)
        dn2 = self.down2(dn1)
        dn3 = self.down3(dn2)
        dn4 = self.down4(dn3)
        dn5 = self.down5(dn4)
        dn6 = self.down6(dn5)
        dn7 = self.down7(dn6)
        dn8 = self.down8(dn7)
        up8 = self.up8(dn8)
        up8 = torch.cat((up8, dn7), 1)
        up7 = self.up7(up8)
        up7 = torch.cat((up7, dn6), 1)
        up6 = self.up6(up7)
        up6 = torch.cat((up6, dn5), 1)
        up5 = self.up5(up6)
        up5 = torch.cat((up5, dn4), 1)
        up4 = self.up4(up5)
        up4 = torch.cat((up4, dn3), 1)
        up3 = self.up3(up4)
        up3 = torch.cat((up3, dn2), 1)
        up2 = self.up2(up3)
        up2 = torch.cat((up2, dn1), 1)
        return self.up1(up2)

### Build the Discriminator class

In [ ]:
class Discriminator(nn.Module):    
    def __init__(self):
        super(Discriminator, self).__init__()
        self.down1 = self.downsample_block(3 * 2, 64, normalize=False)
        self.down2 = self.downsample_block(64, 128)
        self.down3 = self.downsample_block(128, 256)
        self.down4 = self.downsample_block(256, 512)
        self.output = nn.Sequential(nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, 4, padding=1, bias=False))

    def downsample_block(self, in_channels, out_channels, normalize=True):
        layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(negative_slope=0.2))
        return nn.Sequential(*layers)
        
    def forward(self, image1, image2):
        comb_image = torch.cat((image1, image2), 1)
        out = self.down1(comb_image)
        out = self.down2(out)
        out = self.down3(out)
        out = self.down4(out)
        return self.output(out)

In [ ]:
patch_size = (1, 256 // 2 ** 4, 256 // 2 ** 4)

### Train the Model

In [ ]:
adv_loss = torch.nn.MSELoss()
pix_loss = torch.nn.L1Loss()

In [ ]:
n_epochs = 1000
lr = 0.0002
beta_1 = 0.5
beta_2 = 0.999
epoch_show = 20
show_batch = 16
lambda_pix = 100

In [ ]:
gen = Generator().to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))

In [ ]:
summary(gen, input_size = (3, 256, 256), batch_size=-1)

In [ ]:
out_dir = 'Pix2PixOut'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
adv_loss_lst = list()
disc_loss_lst = list()
pixel_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_adv_loss = 0.0
    avg_disc_loss = 0.0
    avg_pixel_loss = 0.0
    num_items = 0
    for real_imgs in tqdm(train_loader):
        cur_batch_size = real_imgs.shape[0]
        real_imgsB = real_imgs[:,:,:,0:256].to(device)
        real_imgsA = real_imgs[:,:,:,256:].to(device)

        valid = torch.ones(cur_batch_size, *patch_size, device=device, requires_grad=False)
        fake = torch.zeros(cur_batch_size, *patch_size, device=device, requires_grad=False)

        #  Train Generator
        gen_opt.zero_grad()
        gen_B = gen(real_imgsA)
        validity = disc(gen_B, real_imgsA)
        galoss = adv_loss(validity, valid)
        gploss = pix_loss(gen_B, real_imgsB)
        gloss = galoss + lambda_pix * gploss
        gloss.backward()
        gen_opt.step()
        
        # Train Discriminator
        disc_opt.zero_grad()
        real_pred = disc(real_imgsB, real_imgsA)
        fake_pred = disc(gen_B.detach(), real_imgsA)
        real_loss = adv_loss(real_pred, valid)
        fake_loss = adv_loss(fake_pred, fake)
        dloss = 0.5 * (real_loss + fake_loss)
        dloss.backward()
        disc_opt.step()
        
        avg_adv_loss += galoss.item() * cur_batch_size
        avg_pixel_loss += gploss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if (epoch + 1) % epoch_show == 0:
        
        show_tensor_images(gen_B.detach(), filename='Pix2PixOut/Pix2PixGridB' + str(epoch) + '.png')
        show_tensor_images(real_imgsA, filename='Pix2PixOut/Pix2PixGridA' + str(epoch) + '.png')
        show_tensor_images(real_imgsB)
        
    ave_adv_loss = avg_adv_loss / num_items
    ave_pixel_loss = avg_pixel_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    adv_loss_lst.append(ave_adv_loss)
    disc_loss_lst.append(ave_disc_loss)
    pixel_loss_lst.append(ave_pixel_loss)
    tqdm_epoch.set_description('Ave Adv Loss: {:5f}, \
                               Ave Disc Loss: {:5f}, \
                               Ave Pixel Loss: {:5f}'.format(ave_adv_loss, ave_disc_loss, ave_pixel_loss))
    torch.save(gen.state_dict(), 'pix2pix_genckpt.pth')
    torch.save(disc.state_dict(), 'pix2pix_discckpt.pth')

In [ ]:
it = iter(test_loader)
sample_img = next(it)

In [ ]:
recon_imgs = gen(sample_img[:,:,:,256:].to(device))

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(sample_img[:,:,:,0:256], num_images=8, 
                       nrow=4, filename='Pix2PixOut/Pix2PixOrigGrid.png')

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(sample_img[:,:,:,256:], num_images=8, 
                       nrow=4, filename='Pix2PixOut/Pix2PixSegGrid.png')

In [ ]:
plt.figure(figsize=(12,12))
show_tensor_images(recon_imgs, num_images=8, 
                       nrow=4, filename='Pix2PixOut/Pix2PixSampleGrid.png')